In [ ]:
import pandas as pd
import seaborn as sn
from dataclasses import dataclass

from src.summarised_result import (
    SummarisedResult,
    SummarisedResultSettings,
    reshape_group_additional,
    reshape_estimate_values,
)

from src.incidence_prevalence import IncidenceResult

# Visualising Incidence and Prevalence results

The plots you get out of the IncidencePrevalence R package are nice.
Let's try to copy them in Python.

First, we need our data.
I have some outputs from the R package in the demo data.

In [ ]:
incidence = pd.read_csv("demo-data/incidence.csv")
point_prevalence = pd.read_csv("demo-data/pointPrevalence.csv")
period_prevalence = pd.read_csv("demo-data/periodPrevalence.csv")

## Incidence (part 1)
First, we need to check what the incidence data look like.

In [ ]:
print(incidence.shape)
incidence.head()

They have done a slightly strange thing, I think to coerce the data into the summarised_result format.
When you `importSummarisedResult` in R, you get out:

```
Rows: 85
Columns: 27
$ X                                    <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, …
$ cdm_name                             <chr> "postgres_omop", "postgres_omop", "postgres_omop", …
$ denominator_cohort_name              <chr> "denominator_cohort_1", "denominator_cohort_1", "de…
$ outcome_cohort_name                  <chr> "neoplasm", "neoplasm", "neoplasm", "neoplasm", "ne…
$ incidence_start_date                 <date> 1927-01-01, 1928-01-01, 1929-01-01, 1930-01-01, 19…
$ incidence_end_date                   <date> 1927-12-31, 1928-12-31, 1929-12-31, 1930-12-31, 19…
$ analysis_interval                    <chr> "years", "years", "years", "years", "years", "years…
$ analysis_censor_cohort_name          <chr> "None", "None", "None", "None", "None", "None", "No…
$ analysis_complete_database_intervals <lgl> TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRU…
$ analysis_outcome_washout             <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
$ analysis_repeated_events             <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FA…
$ denominator_age_group                <chr> "0 to 150", "0 to 150", "0 to 150", "0 to 150", "0 …
$ denominator_days_prior_observation   <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, …
$ denominator_end_date                 <date> 2100-01-01, 2100-01-01, 2100-01-01, 2100-01-01, 21…
$ denominator_requirements_at_entry    <lgl> FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FALSE, FA…
$ denominator_sex                      <chr> "Both", "Both", "Both", "Both", "Both", "Both", "Bo…
$ denominator_start_date               <date> 1900-01-01, 1900-01-01, 1900-01-01, 1900-01-01, 19…
$ denominator_target_cohort_name       <chr> "None", "None", "None", "None", "None", "None", "No…
$ denominator_time_at_risk             <chr> "0 to Inf", "0 to Inf", "0 to Inf", "0 to Inf", "0 …
$ denominator_count                    <int> 10, 20, 31, 34, 44, 57, 69, 80, 88, 99, 110, 117, 1…
$ outcome_count                        <int> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, …
$ person_days                          <int> 2378, 5333, 9354, 12022, 14349, 18342, 23091, 27299…
$ person_years                         <dbl> 6.511, 14.601, 25.610, 32.914, 39.285, 50.218, 63.2…
$ incidence_100000_pys                 <dbl> 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.…
$ incidence_100000_pys_95CI_lower      <dbl> 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.…
$ incidence_100000_pys_95CI_upper      <dbl> 56656.112, 25264.567, 14404.059, 11207.630, 9390.04…
$ result_type                          <chr> "tidy_incidence", "tidy_incidence", "tidy_incidence…
```

So the number of columns has to slightly more than double, and the number of rows has to go down by a factor of seven.
Interesting.

If you look at the `group_name` column, it contains what are two column names in the R table: `denominator_cohort_name` and `outcome_cohort_name`.
The levels of these columns are stored in `group_level`.
There is a similar thing going on with `additional_name` and `additional_level`.

Maybe the thing to do will be to just look at the results where `result_id` is 1.

In [ ]:
incidence.loc[incidence["result_id"] == 1]

Well it's not that.

I'm pretty sure what needs to happen is that the `group_name` has to be split on `"&&&"`, as does `group_level`, and the values of the column names in `group_name` come from `group_level` and the same with `additional_*`.
Then, the `variable_name`, `estimate_name` and `estimate_value` are used in a pivot-y way to reshape the table.

Perhaps I need to read the code for `importSummarisedResult` to get the rules out.

## importSummarisedResult

The code for `importSummarisedResult` is fairly short.
Unfortunately that's because the logic I want isn't there, but in `newSummarisedResult`, which is much more complicated.

```mermaid
graph TD
    importSummarisedResult --> newSummarisedResult
    newSummarisedResult --> constructSummarisedResult
    newSummarisedResult --> validateSummarisedResult
    constructSummarisedResult --> createSettings
    validateSummarisedResult --> validateResultSettings
    validateSummarisedResult --> validateSummarisedResultTable
```

I think I can actually just ignore most of that; I just need to have a bash at it, see if it matches the incidence table.

What helps is that I now know you can read stuff about parsing the table from fields in the table itself.


### Parsing settings

The tables store their settings where the `variable_name` is `"settings"`.

In [ ]:
incidence.loc[incidence["variable_name"] == "settings"]

I think what I need then is a `SummarisedResultSettings` class to work with.
The code for this is part of the package in `src/summarised_result.py`.

In [ ]:
test_settings = SummarisedResultSettings.from_table(
    incidence.loc[(incidence["result_id"] == 1)]
)
test_settings

## Exploding rows

To show how I would normally do something like this, I'll take the first two rows to test on:

In [ ]:
test_exploding_row = incidence.iloc[0:2]
test_exploding_row

Normally I would split the columns containing lists

In [ ]:
test_exploding_row["group_name"].str.split(" &&& ")

then use `.explode` to make multiple rows:

In [ ]:
test_exploding_row["group_name"] = test_exploding_row["group_name"].str.split(" &&& ")
test_exploding_row["group_level"] = test_exploding_row["group_level"].str.split(" &&& ")

exploded_group = test_exploding_row.explode(["group_name", "group_level"])

exploded_group

In [ ]:
exploded_group["additional_name"] = test_exploding_row["additional_name"].str.split(
    " &&& "
)
exploded_group["additional_level"] = test_exploding_row["additional_level"].str.split(
    " &&& "
)

fully_exploded_example = exploded_group.explode(
    ["additional_name", "additional_level"]
).reset_index()
fully_exploded_example

Now we have 6 rows per original row!
Can we simply pivot on those?

In [ ]:
group_example = fully_exploded_example[["group_name", "group_level"]].pivot(
    columns="group_name", values="group_level"
)
group_example

In [ ]:
additional_example = fully_exploded_example[
    ["additional_name", "additional_level"]
].pivot(columns="additional_name", values="additional_level")
additional_example

In [ ]:
fully_exploded_example.join(group_example).join(additional_example).drop(
    ["group_name", "group_level", "additional_name", "additional_level"], axis=1
).set_index("index")

Right, this is nearly it and I'll leave that cell because it shows how it gets built, but I think we can actually join that back to the original table instead.

We can take the exploded rows, join them to the original table's index, then take the first non-`NaN` value for each column to get the target columns for the original table.

In [ ]:
additional_example.join(group_example).join(fully_exploded_example["index"]).set_index(
    "index"
).groupby(level=0).first()

In [ ]:
partly_reshaped_example = reshape_group_additional(
    incidence.loc[
        ~(incidence["variable_name"] == "settings") & (incidence["result_id"] == 1)
    ]
)

partly_reshaped_example

This gets us most of the way there.
The next part is to reshape it in a similar way with estimate_name, estimate_type, and estimate_value.

In [ ]:
estimate_pivot = partly_reshaped_example.pivot(
    columns="estimate_name", values="estimate_value"
)
estimate_pivot

This means we have a table with a bunch of `NaN` again.

In [ ]:
no_estimate_gunk = partly_reshaped_example.drop(
    ["estimate_name", "estimate_type", "estimate_value", "variable_level"], axis=1
)

group_columns = list(no_estimate_gunk.columns)

reshaped_estimate_example = (
    no_estimate_gunk.join(estimate_pivot).groupby(group_columns).first().reset_index()
)

reshaped_estimate_example

This gets us nearly there, though we still have twice as many rows as we need.
Let's look at a time period to see what needs folding in.

In [ ]:
reshaped_estimate_example.loc[
    reshaped_estimate_example["incidence_end_date"] == "1927-12-31"
]

I think, then, it's as simple as

In [ ]:
reshaped_estimate_example.groupby("incidence_start_date").first()

This works! In reality, of course, this final flattening will need to be grouped by more of the variables, but we'll do that sooner by dropping "variable_name", as this is redundant with "denominator" and "outcome" in the `estimate_name`s.

We can group all of this together into one function to get the tables as we want them.

In [ ]:
example_reshaped_estimates = reshape_estimate_values(partly_reshaped_example)
example_reshaped_estimates

OK, we've got the right number of columns here, but not enough columns.

In [ ]:
target_column_names = [
    "X",
    "cdm_name",
    "denominator_cohort_name",
    "outcome_cohort_name",
    "incidence_start_date",
    "incidence_end_date",
    "analysis_interval",
    "analysis_censor_cohort_name",
    "analysis_complete_database_intervals",
    "analysis_outcome_washout",
    "analysis_repeated_events",
    "denominator_age_group",
    "denominator_days_prior_observation",
    "denominator_end_date",
    "denominator_requirements_at_entry",
    "denominator_sex",
    "denominator_start_date",
    "denominator_target_cohort_name",
    "denominator_time_at_risk",
    "denominator_count",
    "outcome_count",
    "person_days",
    "person_years",
    "incidence_100000_pys",
    "incidence_100000_pys_95CI_lower",
    "incidence_100000_pys_95CI_upper",
    "result_type",
]

missing_vars = [x for x in target_column_names if x not in example_reshaped_estimates]
missing_vars

In [ ]:
[x for x in example_reshaped_estimates if x not in target_column_names]

OK, so the target doesn't have `strata_name` and `strata_level`, but I think that should be left in for results where the analysis has stratification.

The parts missing from what we have so far might be in the settings?

In [ ]:
[x for x in test_settings.variables if x.name in missing_vars]

Looks like it!
I think we can populate the table with these, if we need to, but we've got enough to work with now.

## Incidence redux
Now we have a good reason to store the settings and data together.